# RAG Pipeline over a PDF

A retrieval-augmented generation (RAG) pipeline built during my Gen AI
internship at EY. It answers questions about a document by **retrieving** the
most relevant chunks and passing them to an LLM, so answers are grounded in the
document instead of the model's general knowledge.

The internship theme was **Document Intelligence**, and the key question was:
*can RAG run fully locally so sensitive data never leaves the machine, and what
does that cost in speed?* So generation is shown two ways — local Llama 3.1 via
Ollama (private but slow) and Groq-hosted Llama 3.1 (fast but cloud). The
benchmark is at the bottom.

**Flow:** load PDF → split into chunks → embed → store in Chroma → retrieve → answer.

**Stack:** PyMuPDF, LangChain, sentence-transformers (Stella embeddings), Chroma, Llama 3.1 (Ollama / Groq).

In [ ]:
# Install dependencies (run once)
# !pip install -r requirements.txt

## 1. Load the API key

The Groq API key is read from a `.env` file, never hardcoded.
Copy `.env.example` to `.env` and paste your free key from https://console.groq.com.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
assert os.getenv("GROQ_API_KEY"), "Set GROQ_API_KEY in your .env file"

## 2. Load the PDF

PyMuPDF (`fitz`) reads the PDF page by page and pulls out the raw text.
We wrap it in a LangChain `Document` so the splitter can work on it.

In [ ]:
import fitz  # PyMuPDF
from langchain_core.documents import Document

def load_pdf(file_path):
    doc = fitz.open(file_path)
    text = ""
    for page in doc:
        text += page.get_text()
    return text

raw_text = load_pdf("data/sample_docs.pdf")
docs = [Document(page_content=raw_text)]
print(raw_text[:300])

## 3. Split into chunks

The document is too long to embed as one piece, so we split it into chunks.
Embedding smaller chunks means a query retrieves only the most relevant
segments — fewer tokens and more targeted context for the LLM.
- `chunk_size=1000`: small enough for precise retrieval, big enough to keep a full idea together. *(The internship used 4000 for long framework PDFs; 1000 suits this short sample so retrieval has multiple chunks to choose from.)*
- `chunk_overlap=200`: repeats the end of one chunk at the start of the next, so a sentence split across a boundary isn't lost.

In [ ]:
from langchain_text_splitters import CharacterTextSplitter

splitter = CharacterTextSplitter(separator="\n", chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_documents(docs)
print(f"{len(chunks)} chunks")

## 4. Embedding model (local)

Each chunk becomes a vector using the open **Stella** model. It was picked from
the [MTEB leaderboard](https://huggingface.co/spaces/mteb/leaderboard) because it
embeds **locally** — so sensitive document text never goes to a third-party API.
The small adapter class lets LangChain use a sentence-transformers model. The
**same** model must embed both documents and the query, or retrieval breaks.

> `stella_en_400M_v5` (1024-dim) is the efficient variant; the 1.5B version is
> 8192-dim and far heavier. First run downloads ~1.7GB — swap in
> `sentence-transformers/all-MiniLM-L6-v2` (~80MB) to go lighter.

In [ ]:
from sentence_transformers import SentenceTransformer
from langchain_core.embeddings import Embeddings

class SentenceTransformerEmbeddings(Embeddings):
    def __init__(self, model_name):
        self.model = SentenceTransformer(model_name, trust_remote_code=True)
    def embed_documents(self, texts):
        return self.model.encode(texts, convert_to_tensor=True).tolist()
    def embed_query(self, text):
        return self.model.encode(text, convert_to_tensor=True).tolist()

embeddings = SentenceTransformerEmbeddings("bijaygurung/stella_en_400M_v5")

## 5. Store the vectors in Chroma

Chroma is a local, on-disk vector database. It stores each chunk's vector and
lets us search for the chunks closest to a query. No external service or
account needed, which keeps this notebook easy to run.

In [ ]:
from langchain_chroma import Chroma

db = Chroma.from_documents(chunks, embeddings)

## 6. Answer with Groq (fast, cloud)

`RetrievalQA` retrieves the most similar chunks from Chroma and passes them,
plus the question, to the LLM. Groq serves Llama 3.1 on fast inference hardware,
so answers come back in seconds. `temperature=0` keeps answers grounded.

In [ ]:
import time
from langchain_groq import ChatGroq
from langchain.chains import RetrievalQA

llm = ChatGroq(temperature=0, model_name="llama-3.1-8b-instant")
qa = RetrievalQA.from_chain_type(llm, retriever=db.as_retriever())

question = "What are Orion's pricing tiers?"
start = time.time()
result = qa.invoke({"query": question})
print("Answer:", result["result"])
print(f"Time taken: {time.time() - start:.2f}s")

In [ ]:
# Try another question
question = "How is access controlled in Orion?"
print(qa.invoke({"query": question})["result"])

## 7. Same pipeline, fully local (private, slow)

The privacy-preserving alternative: run generation on **local Llama 3.1 via
[Ollama](https://ollama.com)** (`ollama pull llama3.1:8b`) so no data leaves the
machine. Only the LLM changes — retrieval is identical. It's correct but slow on
CPU, which is the whole point of the benchmark below. Uncomment to run locally.

In [ ]:
# from langchain_community.llms import Ollama
#
# local_llm = Ollama(model="llama3.1:8b")
# local_qa = RetrievalQA.from_chain_type(local_llm, retriever=db.as_retriever())
#
# start = time.time()
# print(local_qa.invoke({"query": "Can you summarize the document?"})["result"])
# print(f"Time taken: {time.time() - start:.2f}s")

## Benchmark & takeaways

Measured during the internship (16 GB Windows machine, "summarize the document"):

| Setup | Hardware | Time | Data privacy |
|---|---|---|---|
| Local Stella + local Llama 3.1 | GPU | **8.69 min** | Full |
| Local pipeline (CPU) | CPU | **3.62 min** | Full |
| Groq-hosted Llama 3.1 | API | **2.15 s** | Data sent to cloud |

**Takeaways**
- **Local = full data isolation but too slow for interactive use; Groq is ~200× faster but sends data to the cloud.** For a firm handling client data, that tradeoff is the real decision.
- Retrieval quality matters more than the LLM choice — chunk size and `k` moved answer quality the most.
- Running Llama 3.1 (8B) locally meant CUDA/xformers conflicts, 16 GB RAM crashes, and a ~60 GB model clone.
- **Would improve:** a splitter that respects headings, moving off the legacy `RetrievalQA` to an LCEL chain, and test questions to measure retrieval instead of judging by eye.